In [3]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("Netflix Data new.csv")

# Include N_id alongside core text features
features = ['N_id', 'Title', 'Main Genre', 'Sub Genres', 'Maturity Rating', 'Original Audio', 'Recommendations']

# Filter dataset to selected features and fill missing values
df = df[features].fillna('')

# Convert N_id to string to avoid numeric indexing errors
df['N_id'] = df['N_id'].astype(str)

print("Dataset Loaded Successfully. Shape:", df.shape)
df.head()

Dataset Loaded Successfully. Shape: (6403, 7)


,N_id,Title,Main Genre,Sub Genres,Maturity Rating,Original Audio,Recommendations
0,215309,Ace Ventura: Pet Detective,Comedy,"Comedy, Mystery, US",A,"Hindi, English [Original]","70184054, 60001650, 70112729, 70027007, 115246..."
1,215318,Ace Ventura: When Nature Calls,Comedy,"Comedy, Action & Adventure, US",U/A 16+,"Hindi, English [Original]","70184054, 60001650, 70112729, 70027007, 115246..."
2,217258,The Addams Family,Comedy,"Comedy, US",U/A 13+,"English [Original], Hindi, English - Audio Des...","81156676, 81231974, 70027007, 80049939, 702179..."
3,217303,Addams Family Values,Comedy,"Comedy, US",U/A 13+,"English [Original], Hindi, English - Audio Des...","81156676, 70044593, 81231974, 70027007, 800500..."
4,235527,Agneepath,Drama,"Hindi-Language, Bollywood, Crime, Drama",U/A 16+,Hindi [Original],"17517355, 80158546, 80158395, 80074065, 702042..."


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load Dataset
df = pd.read_csv("Netflix Data new.csv")

# Select relevant columns including N_id and Recommendations
features = ['N_id', 'Title', 'Main Genre', 'Sub Genres', 'Maturity Rating', 'Original Audio', 'Recommendations']
df = df[features].fillna('')

# Convert N_id to string to avoid index type mismatch
df['N_id'] = df['N_id'].astype(str)

# 2. Feature Engineering: Combine features into a single text profile ("soup")
def create_metadata_soup(row):
    return f"{row['Main Genre']} {row['Sub Genres']} {row['Maturity Rating']} {row['Original Audio']}".lower()

df['metadata'] = df.apply(create_metadata_soup, axis=1)

# 3. TF-IDF Vectorization
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['metadata'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

# 4. Compute Cosine Similarity Matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 5. Fast Index Mappings for Lookup
# Map lowercase title -> dataframe index
title_to_idx = pd.Series(df.index, index=df['Title'].str.lower()).drop_duplicates()
# Map N_id -> dataframe index
id_to_idx = pd.Series(df.index, index=df['N_id']).drop_duplicates()

# 6. Recommendation Function
def recommend(query, top_n=5):
    """
    Recommends top_n titles based on Cosine Similarity.
    query: Can be a Title (str) or N_id (str/int)
    """
    query_str = str(query).lower().strip()
    
    # Identify row index by N_id or Title
    if query_str in id_to_idx:
        idx = id_to_idx[query_str]
    elif query_str in title_to_idx:
        idx = title_to_idx[query_str]
    else:
        return f"Error: Item '{query}' not found in dataset."
    
    # Retrieve similarity scores for the target item across all titles
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort titles by similarity score descending (excluding the query item itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    
    # Extract row indices
    recommended_indices = [i[0] for i in sim_scores]
    similarity_values = [round(i[1], 4) for i in sim_scores]
    
    # Output target info and top results
    target_item = df.iloc[idx]
    print(f"--- Recommendations for: '{target_item['Title']}' (ID: {target_item['N_id']}) ---")
    print(f"Genre: {target_item['Main Genre']} | Rating: {target_item['Maturity Rating']}\n")
    
    results = df[['N_id', 'Title', 'Main Genre', 'Sub Genres', 'Maturity Rating']].iloc[recommended_indices].copy()
    results['Similarity Score'] = similarity_values
    
    return results

# ---------------------------------------------------------
# Test the System
# ---------------------------------------------------------
# Test by Title (Replace with an actual title from your CSV)
print(recommend("Stranger Things", top_n=5))

# Test by N_id (Replace with an actual N_id from your CSV)
# print(recommend("101", top_n=5))

TF-IDF Matrix Shape: (6403, 221)
--- Recommendations for: 'Stranger Things' (ID: 80057281) ---
Genre: Sci-Fi | Rating: U/A 16+

          N_id                                  Title Main Genre  \
663   80084447  Shadowhunters: The Mortal Instruments     Sci-Fi   
1986  80996532                          Resident Evil     Sci-Fi   
1099  80174608                   Love, Death & Robots     Sci-Fi   
6364  81937398                               Pantheon     Sci-Fi   
921   80135414                               The Mist     Horror   

                                             Sub Genres Maturity Rating  \
663   Sci-Fi TV, TV Shows Based on Books, Teen TV Sh...         U/A 16+   
1986  Sci-Fi TV, TV Action & Adventure, US TV Shows,...               A   
1099  Sci-Fi TV, TV Action & Adventure, US TV Shows,...               A   
6364  TV Dramas, Sci-Fi TV, TV Shows Based on Books,...         U/A 16+   
921   TV Dramas, Sci-Fi TV, TV Shows Based on Books,...               A   

      Simila